# RAG Data Ingestion Pipeline (MVP)
Building a local vector store (Chroma) for Retrieval-Augmented Generation.

The pipeline loads all PDFs from the `Rag Database` directory, splits them into chunks,
generates embeddings via Google Gemini, and stores everything in a persistent
Chroma DB under `./chroma_db`.

**Dependencies:** `langchain`, `langchain-community`, `langchain-google-genai`,
`langchain-text-splitters`, `pypdf`, `chromadb`, `python-dotenv`

## 1) SETUP – load API key from .env
Same mechanism as in `agent.ipynb` / `app.py`: `GEMINI_API_KEY` is loaded via
`python-dotenv` and validated before use.

In [2]:
import os
from glob import glob
from dotenv import load_dotenv

# Load API key from .env (same approach as in agent.ipynb / app.py)
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found in .env!")

os.environ["GOOGLE_API_KEY"] = api_key  # for langchain-google-genai
print("Setup complete. GEMINI_API_KEY loaded.")

Setup complete. GEMINI_API_KEY loaded.


## 2) DOCUMENT LOADING – PDFs + Markdown with bucket structure
Loads all `*.pdf` and `*.md` files. The directory determines the bucket (`box`) metadata:

- **Root** (`Rag Database`) → `box=1` (default)
- `Rag Database/box1_patterns/` → `box=1`
- `Rag Database/box2_domain/` → `box=2`

PDFs are loaded page-by-page with `PyPDFLoader` (source = path, as before); Markdown
files are loaded as plain text with `TextLoader` (source = filename, page = 0). Every chunk
receives metadata `box` (int). Bucket sub-directories that do not yet exist are skipped.

In [3]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = "Rag Database"

# Directory name -> bucket number. Files in the root default to box 1.
BUCKET_DIRS = {
    "box1_patterns": 1,
    "box2_domain": 2,
}


def load_dir(directory, box):
    """Load all PDFs and Markdown files in *directory*, tagging metadata.

    PDFs keep PyPDFLoader behaviour (source = path, one Document per page);
    Markdown files are loaded as plain text with source = filename, page = 0.
    Each Document receives metadata box=<box>.
    """
    docs = []
    for pdf_path in sorted(glob(os.path.join(directory, "*.pdf"))):
        docs.extend(PyPDFLoader(pdf_path).load())
    for md_path in sorted(glob(os.path.join(directory, "*.md"))):
        for d in TextLoader(md_path, encoding="utf-8").load():
            d.metadata["source"] = os.path.basename(md_path)
            d.metadata["page"] = 0
            docs.append(d)
    for d in docs:
        d.metadata["box"] = box
    return docs


documents = []
documents.extend(load_dir(DATA_DIR, 1))  # root -> box 1 default
for dirname, box in BUCKET_DIRS.items():
    subdir = os.path.join(DATA_DIR, dirname)
    if os.path.isdir(subdir):
        documents.extend(load_dir(subdir, box))

if not documents:
    raise FileNotFoundError(f"No PDFs or Markdown files found under '{DATA_DIR}'!")

boxes = {}
for d in documents:
    boxes[d.metadata["box"]] = boxes.get(d.metadata["box"], 0) + 1

print(f"Loaded documents: {len(documents)}  (by box: {boxes})")

C:\Users\Kusha\AppData\Local\Temp\ipykernel_11776\606662502.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader
incorrect startxref pointer(1)
parsing for Object Streams


Loaded documents: 172  (by box: {1: 172})


## 3) CHUNKING – split text with RecursiveCharacterTextSplitter
The recursive splitter divides the text along natural separators (paragraph, sentence,
word) so that related content stays within a single chunk as much as possible.

- `chunk_size = 1000` (characters per chunk)
- `chunk_overlap = 200` (overlap so context at the edges is preserved)

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(documents)
print(f"Created chunks: {len(chunks)}")
print(f"Example metadata of a chunk: {chunks[0].metadata}")

Created chunks: 473
Example metadata of a chunk: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-29T20:34:32+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-29T20:34:32+00:00', 'subject': '(unspecified)', 'title': 'Architecture Patterns - Bucket 1 Knowledge Base', 'trapped': '/False', 'source': 'Rag Database\\architecture_patterns.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'box': 1}


## 4) EMBEDDINGS & VECTOR STORE – create and persist the Chroma database
Embeddings via Google's model `models/gemini-embedding-2`.

**Note on the free-tier quota:** The embedding quota is 100 requests/min.
With a shared/rolled-over API key this window is often briefly exhausted,
which can trigger `429 RESOURCE_EXHAUSTED`. That is why the chunks are inserted
here in batches with retry + backoff – this is more robust than a single
`from_documents` call. The finished DB lives in `./chroma_db` and can later be
loaded without re-embedding.

**Idempotency:** Before inserting, an existing collection is deleted via
`delete_collection()`, so that running the notebook multiple times does not
duplicate the vectors (more robust on Windows than deleting the directory).

In [5]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

CHROMA_DIR = "./chroma_db"
BATCH_SIZE = 100  # Google's embedding API allows max. 100 texts per request
MAX_TRIES = 6     # retry attempts per batch on 429 (quota exceeded)

def add_with_retry(vs, docs):
    """Inserts a batch; retries on transient errors (429/5xx) with backoff."""
    transient = ("429", "500", "502", "503", "504", "RESOURCE_EXHAUSTED", "Bad Gateway", "getaddrinfo", "ConnectError", "timed out")
    for attempt in range(1, MAX_TRIES + 1):
        try:
            vs.add_documents(docs)
            return
        except Exception as e:
            msg = str(e)
            if any(code in msg for code in transient) and attempt < MAX_TRIES:
                wait = 30 * attempt
                print(f"   Transient error -> waiting {wait}s (attempt {attempt}/{MAX_TRIES - 1})...")
                time.sleep(wait)
            else:
                raise

# Idempotency: delete the existing collection so that running multiple times does
# not create duplicates (via the Chroma API instead of shutil.rmtree = more robust
# on Windows, since Chroma otherwise exclusively locks the index files).
_purge = Chroma(embedding_function=embeddings, persist_directory=CHROMA_DIR)
try:
    _purge.delete_collection()
    print("   Existing collection deleted.")
except Exception:
    print("   No existing collection – fresh build.")

# Create a fresh (persistent) collection and fill it in batches
vectorstore = Chroma(embedding_function=embeddings, persist_directory=CHROMA_DIR)

total = len(chunks)
for start in range(0, total, BATCH_SIZE):
    batch = chunks[start:start + BATCH_SIZE]
    add_with_retry(vectorstore, batch)
    done = min(start + BATCH_SIZE, total)
    print(f"   Inserted: {done}/{total} chunks")

print(f"\nVector store saved under: {CHROMA_DIR}")
print(f"Indexed vectors: {vectorstore._collection.count()}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
C:\Users\Kusha\AppData\Local\Temp\ipykernel_11776\1199454109.py:30: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  _purge = Chroma(embedding_function=embeddings, persist_directory=CHROMA_DIR)


   Existing collection deleted.
   Inserted: 100/473 chunks
   Transient error -> waiting 30s (attempt 1/5)...
   Transient error -> waiting 60s (attempt 2/5)...
   Inserted: 200/473 chunks
   Transient error -> waiting 30s (attempt 1/5)...
   Inserted: 300/473 chunks
   Transient error -> waiting 30s (attempt 1/5)...
   Transient error -> waiting 60s (attempt 2/5)...
   Inserted: 400/473 chunks
   Transient error -> waiting 30s (attempt 1/5)...
   Inserted: 473/473 chunks

Vector store saved under: ./chroma_db
Indexed vectors: 473


## 5) TEST QUERY – similarity search (Microservices)
An architecture-related test query against the vector store. The top-3 hits are
printed together with their source PDF (metadata).

Note: The DB is already persisted – for later searches it can be loaded without
re-embedding:
```python
vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
```

In [6]:
query = "What are the benefits of a microservices architecture?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results, 1):
    source = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "?")
    print(f"--- Result {i} | Source: {source} | Page: {page} ---")
    print(doc.page_content[:300].strip())
    print()

Query: What are the benefits of a microservices architecture?

--- Result 1 | Source: architecture_patterns_v2.md | Page: 0 ---
**Trade-offs:** You gain strong module boundaries, independent deployability, and independent scalability; you pay the "Microservice Premium" — distribution (remote calls are slow and can fail), eventual consistency, and operational complexity requiring a mature ops team. Fowler documents a case whe

--- Result 2 | Source: Rag Database\microservices-on-aws.pdf | Page: 23 ---
Implementing Microservices on AWS AWS Whitepaper
Cost optimization and sustainability
Microservices architecture can enhance cost optimization and sustainability. By breaking an 
application into smaller parts, you can scale up only the services that need more resources, 
reducing cost and waste. Th

--- Result 3 | Source: Rag Database\architecture_patterns.pdf | Page: 1 ---
Structural
Pattern: Microservices
Problem: A large monolith blocks independent deployment, scaling, and team autonom